# M6: supplementary itinerary judge

Prepared, not executed. This notebook evaluates existing plans only. It does not rerun or change M6. Read README.md in this folder before enabling live requests.

إضافة تقييم مستقلة للخطط المحفوظة، دون تغيير نتائج M6 الأصلية. التشغيل الحي مغلق افتراضيًا. لا توجد درجات جديدة بعد.

In [ ]:
from pathlib import Path
import subprocess

# Optional checkout into a fresh Colab runtime. No installation or model calls here.
DOWNLOAD_REPOSITORY = False
project = Path('/content/riyadh-travel-planner-m6-multi-agent')
if DOWNLOAD_REPOSITORY and not project.exists():
    subprocess.run(['git', 'clone', '--single-branch', '--branch', 'm6/implementation',
                    'https://github.com/BashaierAwal/riyadh-travel-planner-m6-multi-agent.git',
                    str(project)], check=True)
if not project.exists():
    raise RuntimeError('Set DOWNLOAD_REPOSITORY=True in Colab, or set project to your local checkout.')


## 1. Offline preparation
Expected: 120 records, 59 readable baseline plans and 55 readable treatment plans. No model key required.

In [ ]:
import sys
subprocess.run([sys.executable, '-B', 'evaluation/llm_judge/evaluate.py'], cwd=project, check=True)
subprocess.run([sys.executable, '-B', '-m', 'unittest', 'discover', '-s',
                'evaluation/llm_judge', '-p', 'test_*.py', '-v'], cwd=project, check=True)


## 2. Optional dependency setup
Installation is disabled. These are the existing pinned project dependencies, not a model run.

In [ ]:
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt',
                    '-r', 'requirements-gemini.txt'], cwd=project, check=True)


## 3. Live judge — disabled
First agree the model, request limit and account usage. Check that the model is available; do not silently substitute it. Preserve the output folder in persistent storage after every batch and restore it before resuming. A separate pilot version must not be pooled into formal results. Never run simultaneous sessions for the same version.

المرحلة التالية قد تستهلك حصة Gemini. لا تفعّليها قبل الاتفاق على النموذج وحد الطلبات والحفظ الدائم.

In [ ]:
RUN_LIVE = False
JUDGE_MODEL = ''  # Set only after confirming the evaluation model.
MAX_REQUESTS = 0  # Explicit per-invocation cap; start with an approved small pilot.
VERSION = 'judge-pilot-v1'
if RUN_LIVE:
    if not JUDGE_MODEL or not 1 <= MAX_REQUESTS <= 114:
        raise ValueError('Choose an approved model and request cap first.')
    import os
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    try:
        subprocess.run([sys.executable, '-B', 'evaluation/llm_judge/evaluate.py', '--run',
                        '--approve-live', '--model', JUDGE_MODEL, '--max-requests',
                        str(MAX_REQUESTS), '--version', VERSION], cwd=project, check=True)
    finally:
        os.environ.pop('GEMINI_API_KEY', None)
else:
    print('Live judge disabled. No requests sent.')


## 4. Read saved status
Means apply only to judged plans. Generation failures and judge failures remain separate. A partial summary is not a completed comparison. Review model reasons before reporting conclusions.

In [ ]:
import json
summary = project / 'evaluation/llm_judge/runs' / VERSION / 'summary.json'
if summary.exists():
    print(json.dumps(json.loads(summary.read_text()), indent=2))
else:
    print('No judge results yet.')
